# Vuelos — Consultas SQL

Práctica 5, Ejercicio 2 de Introducción a Base de Datos (UNLP). Las tablas se crean en una base DuckDB en memoria con datos de ejemplo inventados, y cada consulta se ejecuta mostrando su resultado.

Las consultas se ejecutan en orden: la 6 elimina el vuelo LOM3524, así que las siguientes ya no lo ven.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from consultas import crear_base, cargar_consultas, ejecutar

con = crear_base("vuelos")
consultas = cargar_consultas("vuelos")

In [2]:
# Cantidad de filas de cada tabla
con.sql("SELECT table_name AS tabla, estimated_size AS filas FROM duckdb_tables() ORDER BY table_name").df()

,tabla,filas
0,Aerolinea,3
1,Ciudad,5
2,Cliente,5
3,Pasaje,15
4,Vuelo,7


## Consulta 1

**Qué cambió respecto de mi resolución en papel:** faltaba cerrar los paréntesis de los dos `NOT EXISTS`. Saqué el `DISTINCT` porque la consulta de afuera no tiene `JOIN` y no puede repetir clientes.

In [3]:
ejecutar(con, consultas, 1)

1. Clientes que viajaron con todas las aerolíneas

SELECT c.nombre, c.apellido, c.pasaporte, c.nacionalidad
FROM Cliente c
WHERE NOT EXISTS (
    SELECT *
    FROM Aerolinea a
    WHERE NOT EXISTS (
        SELECT *
        FROM Pasaje p
        INNER JOIN Vuelo v ON (v.codVuelo = p.codVuelo)
        WHERE (c.codCliente = p.codCliente) AND (v.codAerolinea = a.codAerolinea)
    )
); 



,nombre,apellido,pasaporte,nacionalidad
0,Juan,Pérez,AAA111111,Argentina
1,Lucía,Gómez,AAA333333,Argentina


## Consulta 2

**Qué cambió respecto de mi resolución en papel:** Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`). Agregué `codCliente` al `GROUP BY`: sin él, dos clientes con el mismo nombre y apellido se sumarían como si fueran uno.

In [4]:
ejecutar(con, consultas, 2)

2. Cantidad de viajes con destino Buenos Aires que hizo cada cliente

SELECT c.nombre, c.apellido, c.pasaporte, c.nacionalidad, COUNT(*) AS cantidad
FROM Cliente c
INNER JOIN Pasaje p ON (p.codCliente = c.codCliente)
INNER JOIN Vuelo v ON (v.codVuelo = p.codVuelo)
INNER JOIN Ciudad ciu ON (v.cod_ciudad_destino = ciu.codCiudad)
WHERE (ciu.nombre = 'Buenos Aires')
GROUP BY c.codCliente, c.nombre, c.apellido, c.pasaporte, c.nacionalidad
ORDER BY cantidad DESC, c.apellido; 



,nombre,apellido,pasaporte,nacionalidad,cantidad
0,Lucía,Gómez,AAA333333,Argentina,2
1,Juan,Pérez,AAA111111,Argentina,1
2,Olena,Shevchenko,UKR222222,Ucrania,1
3,Pedro,Silva,BRA444444,Brasil,1
4,Ana,Torres,AAA555555,Argentina,1


## Consulta 3

**Qué cambió respecto de mi resolución en papel:** al `JOIN` de `Ciudad cd` le faltaba el `ON`, los alias estaban cruzados (`co` estaba unido al destino) y el `GROUP BY` usaba `ciu`, que no existe en esta consulta.

In [5]:
ejecutar(con, consultas, 3)

3. Vuelos en los que se vendieron más pasajes que su capacidad (sobreventa)

SELECT v.codVuelo, v.fecha, v.hora, co.nombre AS origen, cd.nombre AS destino,
       v.cantidad_pasajes, COUNT(p.codReserva) AS vendidos
FROM Vuelo v
INNER JOIN Ciudad co ON (v.cod_ciudad_origen = co.codCiudad)
INNER JOIN Ciudad cd ON (v.cod_ciudad_destino = cd.codCiudad)
INNER JOIN Pasaje p ON (v.codVuelo = p.codVuelo)
GROUP BY v.codVuelo, v.fecha, v.hora, co.nombre, cd.nombre, v.cantidad_pasajes
HAVING COUNT(p.codReserva) > v.cantidad_pasajes; 



,codVuelo,fecha,hora,origen,destino,cantidad_pasajes,vendidos
0,AR1001,2018-03-10,08:30:00,Córdoba,Buenos Aires,3,4


## Consulta 4

**Qué cambió respecto de mi resolución en papel:** faltaba cerrar un paréntesis. En la subconsulta cambié los alias a `p2` y `v2`, para no confundirlos con los de afuera.

In [6]:
ejecutar(con, consultas, 4)

4. Clientes que viajaron solamente en 2018, ordenados por apellido y nombre

SELECT DISTINCT c.nombre, c.apellido, c.pasaporte, c.nacionalidad
FROM Cliente c
INNER JOIN Pasaje p ON (c.codCliente = p.codCliente)
INNER JOIN Vuelo v ON (p.codVuelo = v.codVuelo)
WHERE (YEAR(v.fecha) = 2018) AND (c.codCliente NOT IN (
    SELECT p2.codCliente
    FROM Pasaje p2
    INNER JOIN Vuelo v2 ON (p2.codVuelo = v2.codVuelo)
    WHERE (YEAR(v2.fecha) <> 2018)
))
ORDER BY c.apellido ASC, c.nombre ASC; 



,nombre,apellido,pasaporte,nacionalidad
0,Pedro,Silva,BRA444444,Brasil


## Consulta 5

**Qué cambió respecto de mi resolución en papel:** estaba bien. Agregué `codCiudad` al `GROUP BY` por si hay dos ciudades con el mismo nombre en distintos países.

In [7]:
ejecutar(con, consultas, 5)

5. Cantidad de pasajeros que llegó a cada ciudad durante 2018

SELECT ciu.nombre, ciu.pais, COUNT(p.codCliente) AS pasajeros
FROM Ciudad ciu
INNER JOIN Vuelo v ON (ciu.codCiudad = v.cod_ciudad_destino)
INNER JOIN Pasaje p ON (p.codVuelo = v.codVuelo)
WHERE (YEAR(v.fecha) = 2018)
GROUP BY ciu.codCiudad, ciu.nombre, ciu.pais
ORDER BY pasajeros DESC; 



,nombre,pais,pasajeros
0,Buenos Aires,Argentina,4
1,Cancún,México,3
2,Salta,Argentina,2


## Consulta 6

**Qué cambió respecto de mi resolución en papel:** solo borraba el vuelo, pero sus pasajes lo referencian con una clave foránea, así que el motor no lo deja borrar (ver al final). Primero se borran los pasajes y después el vuelo, igual que hice con `Detalle_Reserva` y `Reserva` en los parciales.

In [8]:
ejecutar(con, consultas, 6)

6. Eliminar el vuelo LOM3524 (primero sus pasajes, porque lo referencian)

DELETE FROM Pasaje
WHERE (codVuelo = 'LOM3524');

DELETE FROM Vuelo
WHERE (codVuelo = 'LOM3524'); 

Filas afectadas: 2
Filas afectadas: 1


## Consulta 7

**Qué cambió respecto de mi resolución en papel:** faltaba cerrar un paréntesis. Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`).

In [9]:
ejecutar(con, consultas, 7)

7. Clientes que viajaron a Cancún en 2018 y no volvieron a viajar en 2019

SELECT DISTINCT c.nombre, c.apellido, c.pasaporte, c.nacionalidad
FROM Cliente c
INNER JOIN Pasaje p ON (p.codCliente = c.codCliente)
INNER JOIN Vuelo v ON (v.codVuelo = p.codVuelo)
INNER JOIN Ciudad ciu ON (ciu.codCiudad = v.cod_ciudad_destino)
WHERE (YEAR(v.fecha) = 2018) AND (ciu.nombre = 'Cancún') AND (c.codCliente NOT IN (
    SELECT p2.codCliente
    FROM Pasaje p2
    INNER JOIN Vuelo v2 ON (v2.codVuelo = p2.codVuelo)
    WHERE (YEAR(v2.fecha) = 2019)
)); 



,nombre,apellido,pasaporte,nacionalidad
0,Pedro,Silva,BRA444444,Brasil


## Consulta 8

**Qué cambió respecto de mi resolución en papel:** Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`). Agregué `codVuelo` a las columnas, porque `UNION` elimina filas iguales y dos vuelos distintos con los mismos datos se juntarían en uno.

In [10]:
ejecutar(con, consultas, 8)

8. Vuelos con algún pasajero de nacionalidad ucraniana o con destino Buenos Aires

SELECT v.codVuelo, v.fecha, v.hora, v.cod_ciudad_origen, v.cod_ciudad_destino, v.cantidad_pasajes, v.codAerolinea
FROM Vuelo v
INNER JOIN Pasaje p ON (v.codVuelo = p.codVuelo)
INNER JOIN Cliente c ON (p.codCliente = c.codCliente)
WHERE (c.nacionalidad = 'Ucrania')
UNION
SELECT v.codVuelo, v.fecha, v.hora, v.cod_ciudad_origen, v.cod_ciudad_destino, v.cantidad_pasajes, v.codAerolinea
FROM Vuelo v
INNER JOIN Ciudad ciu ON (v.cod_ciudad_destino = ciu.codCiudad)
WHERE (ciu.nombre = 'Buenos Aires')
ORDER BY fecha; 



,codVuelo,fecha,hora,cod_ciudad_origen,cod_ciudad_destino,cantidad_pasajes,codAerolinea
0,AR1001,2018-03-10,08:30:00,4,1,3,1
1,AR1002,2018-07-22,14:00:00,1,2,150,1


## Consulta 9

**Qué cambió respecto de mi resolución en papel:** Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`).

In [11]:
ejecutar(con, consultas, 9)

9. Clientes que viajaron a Salta y también a Jujuy

SELECT c.nombre, c.apellido, c.pasaporte, c.nacionalidad
FROM Cliente c
INNER JOIN Pasaje p ON (c.codCliente = p.codCliente)
INNER JOIN Vuelo v ON (v.codVuelo = p.codVuelo)
INNER JOIN Ciudad ciu ON (v.cod_ciudad_destino = ciu.codCiudad)
WHERE (ciu.nombre = 'Salta')
INTERSECT
SELECT c.nombre, c.apellido, c.pasaporte, c.nacionalidad
FROM Cliente c
INNER JOIN Pasaje p ON (c.codCliente = p.codCliente)
INNER JOIN Vuelo v ON (v.codVuelo = p.codVuelo)
INNER JOIN Ciudad ciu ON (v.cod_ciudad_destino = ciu.codCiudad)
WHERE (ciu.nombre = 'Jujuy'); 



,nombre,apellido,pasaporte,nacionalidad
0,Ana,Torres,AAA555555,Argentina


## Consulta 10

**Qué cambió respecto de mi resolución en papel:** faltaba cerrar un paréntesis. Las comillas tipográficas que pone Word (“ ” ‘ ’) no son válidas en SQL: los textos van entre comillas simples rectas (`'...'`).

In [12]:
ejecutar(con, consultas, 10)

10. Aerolíneas que solo vuelan a destinos dentro de Argentina

SELECT DISTINCT a.nombre, a.origen
FROM Aerolinea a
INNER JOIN Vuelo v ON (v.codAerolinea = a.codAerolinea)
INNER JOIN Ciudad ciu ON (v.cod_ciudad_destino = ciu.codCiudad)
WHERE (ciu.pais = 'Argentina') AND (a.codAerolinea NOT IN (
    SELECT v2.codAerolinea
    FROM Vuelo v2
    INNER JOIN Ciudad ciu2 ON (v2.cod_ciudad_destino = ciu2.codCiudad)
    WHERE (ciu2.pais <> 'Argentina')
)); 



,nombre,origen
0,Aerolíneas Argentinas,Argentina
1,Flybondi,Argentina


## Por qué importaban las correcciones

Estas celdas ejecutan algunas versiones originales sobre una base nueva, para ver qué pasaba sin la corrección.

**Consulta 6 sin borrar primero los pasajes:** la clave foránea lo impide.

In [13]:
base = crear_base("vuelos")
try:
    base.execute("DELETE FROM Vuelo WHERE (codVuelo = 'LOM3524')")
except Exception as error:
    print(type(error).__name__, "->", str(error).split(". ")[0])

ConstraintException -> Constraint Error: Violates foreign key constraint because key "codVuelo: LOM3524" is still referenced by a foreign key in a different table
